# ЛР1 — аудиторный demo-notebook
## NumPy для нейронных сетей: формы → `@` → broadcasting → `X @ W + b`

**Назначение:** короткий notebook для live-разбора на первой практике. Это **не** сдаваемая ЛР1 и не содержит решений её основных TODO.

Во время занятия действует правило: **сначала прогнозируем результат или форму — потом запускаем ячейку**.

## 0. Старт и проверка окружения

Для ЛР1 достаточно Python и NumPy. GPU не нужен.

Локально классический Jupyter Notebook можно установить и запустить так:

```bash
python -m pip install notebook numpy
jupyter notebook
```

Альтернатива — открыть `.ipynb` в Google Colab через **File → Upload notebook**.

In [ ]:
import sys
import numpy as np

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)

### Мини-экскурс в Jupyter


Запустите следующую ячейку несколько раз. Значение меняется, потому что notebook хранит **состояние выполнения**, а не только текст кода.

In [ ]:
state_demo = 10
print("Сначала:", state_demo)

In [ ]:
state_demo += 1
print("После повторного запуска этой ячейки:", state_demo)

> После демонстрации: **Restart kernel → Run All**. Это простой способ проверить, что notebook воспроизводим сверху вниз.

# 1. Сначала — обычная таблица

Не вводим буквы `B`, `D`, `H` сразу. Начинаем с формулировки:

> «Перед нами таблица: **3 строки и 2 столбца**. Что может означать одна строка? А один столбец?»


In [ ]:
X = np.array([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0],
])

print(X)
print("shape:", X.shape)
print("первая строка:", X[0])
print("первый столбец:", X[:, 0])

**Вопросы до следующего запуска**

- Что вернёт сумма по `axis=0`: одно число, два числа или три?
- А по `axis=1`?


In [ ]:
print("sum(axis=0):", X.sum(axis=0), "shape =", X.sum(axis=0).shape)
print("sum(axis=1):", X.sum(axis=1), "shape =", X.sum(axis=1).shape)

# 2. `*` и `@` — две разные идеи

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

print("a * b =", a * b)
print("a @ b =", a @ b)
print("sum(a * b) =", np.sum(a * b))

# 3. Из 2 столбцов хотим получить 4 новых


> «У нас три строки. В каждой строке два исходных числа. Хотим для каждой строки вычислить **четыре новых числа**.»

Для этого нужна таблица коэффициентов `W` размера **2 × 4**.

Только теперь вводим обозначения:

- `B = 3` — число строк / размер batch;
- `D = 2` — число исходных столбцов / входных признаков;
- `H = 4` — число новых столбцов / выходных признаков слоя.

Правило форм:

`(B, D) @ (D, H) → (B, H)`

In [ ]:
W = np.array([
    [ 0.5,  1.0, -1.0,  0.0],
    [ 2.0, -0.5,  0.5,  1.0],
])

Y = X @ W

print("X:", X.shape)
print("W:", W.shape)
print("X @ W:", Y.shape)
print("\nРезультат:\n", Y)

### Что означает `W`

Каждый **столбец `W`** — отдельный набор коэффициентов для вычисления одного нового выходного числа.

Проверим вручную только один элемент: первое новое число для первой строки.

In [ ]:
manual = X[0, 0] * W[0, 0] + X[0, 1] * W[1, 0]
print("вручную:", manual)
print("через X @ W:", Y[0, 0])

# 4. Bias и broadcasting

Теперь у нас есть таблица `Y` размера **3 × 4**. Хотим к каждому из четырёх новых столбцов добавить свою небольшую поправку.

Вместо копирования одной и той же строки три раза NumPy позволяет прибавить вектор длины 4 ко всем строкам сразу.

In [ ]:
bias = np.array([0.1, -0.2, 0.3, 0.0])
Z = Y + bias

print("Y:", Y.shape)
print("bias:", bias.shape)
print("Z:", Z.shape)
print("\nZ:\n", Z)

### Намеренно неподходящая форма

In [ ]:
bad_bias = np.array([0.1, 0.2, 0.3])

try:
    Y + bad_bias
except ValueError as err:
    print(type(err).__name__ + ":", err)

# 5. Линейный слой — это уже знакомые кирпичики

Ничего принципиально нового не добавляем. Собираем вместе матричное умножение и broadcasting:

`X @ W + b`

In [ ]:
def linear_batch(X, W, b):
    return X @ W + b

Z = linear_batch(X, W, bias)
print("форма выхода слоя:", Z.shape)

# 6. Мини-checkpoint

Теперь новые размеры — без копирования предыдущего примера:

- 5 строк;
- 3 исходных столбца;
- хотим получить 2 новых столбца.

**Задание:** создать `X_check`, `W_check`, `b_check`, получить `Z_check = X_check @ W_check + b_check` и вывести формы всех массивов.

**До кода** сказать формы `X`, `W`, `b` и `Z`.

In [ ]:
# B = 5, D = 3, H = 2
#
# X_check = ...
# W_check = ...
# b_check = ...
# Z_check = ...


# 7. Тизер домашней части: большие logits

In [ ]:
logits_big = np.array([[1000.0, 1001.0, 1002.0]])

with np.errstate(over="ignore"):
    exp_values = np.exp(logits_big)

print("exp(logits) =", exp_values)
print("все значения конечны?", np.isfinite(exp_values).all())